In [ ]:
import pandas as pd
from src.configs.config import settings
import re

df_corpus = pd.read_json(settings.DOCUMENT_COLLECTION_PATH)
df_queries = pd.read_json(settings.QUERIES_COLLECTION_PATH)
df_pq = pd.read_parquet('./data/0000.parquet')

def clean_article_text(raw_text: str) -> str:
    if not isinstance(raw_text, str):
        return ""    
    cleaned = raw_text
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    return cleaned.strip()

df_corpus['body'] = df_corpus['body'].apply(clean_article_text)
df_corpus['published_at'] = df_corpus['published_at'].astype('str')


In [ ]:
# 1. Grab 10 random articles from your cleaned DataFrame
sample_df = df_corpus.sample(n=10, random_state=10)

# 2. Compile them into a single string
sample_text = ""
for index, row in sample_df.iterrows():
    sample_text += f"--- ARTICLE {index} ---\n{row['body']}\n\n"

# 3. Save it to a text file
with open("./data/schema_discovery_sample6.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)
    
print("Sample saved! Open schema_discovery_sample6.txt")

In [ ]:

import pandas as pd
from src.bots.graph_retrieval_v3 import GraphRAGRetriever
from src.configs.config import settings
from src.bots.models import ask_vector_rag_direct
import time
from src.db.graph_store import graph_db
from src.db.vector_store import chroma_db
from src.bots.hybrid_retrieval import HybridGraphVectorRetriever

graph_retriever = GraphRAGRetriever(graph_db, settings.OPENAI_API_KEY)
hybrid_retriever = HybridGraphVectorRetriever(graph_db, chroma_db, settings.OPENAI_API_KEY)


path_to_test_set = "./results/test_set.json"
df_test = pd.read_json(path_to_test_set, orient="records")

for index, item in df_test.iterrows():
    query = item['query']
    
    if item['vector_rag_status'] == 'PENDING':
        print(f"\n[{index}] Vector RAG processing...")
        try:
            start_time = time.perf_counter()
            answer = ask_vector_rag_direct(query)
            
            latency = time.perf_counter() - start_time
            
            df_test.at[index, 'vector_rag_answer'] = answer
            df_test.at[index, 'vector_rag_latency'] = round(latency, 2)
            df_test.at[index, 'vector_rag_status'] = 'SUCCESS'
            
        except Exception as e:
            print(f"Vector RAG Error: {str(e)}")
            df_test.at[index, 'vector_rag_status'] = 'FAILED'

    if item['graph_rag_status'] == 'PENDING':
        print(f"[{index}] Graph RAG processing...")
        try:
            start_time = time.perf_counter()
            
            answer = graph_retriever.answer(query, verbose=True)
            
            latency = time.perf_counter() - start_time
            print(f'Graph RAG answer: {answer}')
        
            df_test.at[index, 'graph_rag_answer'] = answer
            df_test.at[index, 'graph_rag_latency'] = round(latency, 2)
            df_test.at[index, 'graph_rag_status'] = 'SUCCESS'
        
        except Exception as e:
            print(f"Graph RAG Error: {str(e)}")
            df_test.at[index, 'graph_rag_status'] = 'FAILED'
    
    if item['hybrid_rag_status'] == 'PENDING':
        print(f"[{index}] Hybrid RAG processing...")
        try:
            start_time = time.perf_counter()
            
            answer = hybrid_retriever.answer(query, verbose=True)
            
            latency = time.perf_counter() - start_time
            print(f'Hybrid RAG answer: {answer}')
        
            df_test.at[index, 'hybrid_rag_answer'] = answer
            df_test.at[index, 'hybrid_rag_latency'] = round(latency, 2)
            df_test.at[index, 'hybrid_rag_status'] = 'SUCCESS'
        
        except Exception as e:
            print(f"Hybrid RAG Error: {str(e)}")
            df_test.at[index, 'hybrid_rag_status'] = 'FAILED'
    
    df_test.to_json(path_to_test_set, orient="records", indent=2)


In [ ]:
from src.scripts.common import upsert_test_set
import pandas as pd
from src.configs.config import settings

df_queries = pd.read_json('./data/sample_queries.json')
path_to_test_set = "./results/test_set.json"
# upsert_test_set(df_queries, path_to_test_set, 50)
path_to_test_set = "./results/test_set.json"
dd = pd.read_json(path_to_test_set, orient="records")
dd
dd[['question_type','answer','vector_rag_answer','graph_rag_answer','hybrid_rag_answer', 'graph_rag_status']].iloc[-60:]
# for im, r in df_queries[df_queries['question_type'] == 'null_query'].iterrows():
#     if im%17 == 0:
#         print(r['query'])

In [ ]:
from src.scripts.common import upsert_test_set
import pandas as pd
from src.configs.config import settings

df_queries = pd.read_json('./data/sample_queries.json')
path_to_test_set = "./results/test_set.json"
# upsert_test_set(df_queries, path_to_test_set, 50)
path_to_test_set = "./results/test_set.json"
dd = pd.read_json(path_to_test_set, orient="records")
dd
dd[['question_type','answer','vector_rag_answer','graph_rag_answer','hybrid_rag_answer', 'graph_rag_status']].iloc[-60:]

In [ ]:
import pandas as pd
from src.bots.graph_retrieval_v3 import GraphRAGRetriever
from src.configs.config import settings
from src.db.graph_store import graph_db
from src.db.vector_store import chroma_db
from src.bots.hybrid_retrieval import HybridGraphVectorRetriever

graph_retriever = GraphRAGRetriever(graph_db, settings.OPENAI_API_KEY)
# hybrid_retriever = HybridGraphVectorRetriever(graph_db, chroma_db, settings.OPENAI_API_KEY)
path_to_test_set = "./results/test_set.json"
dd = pd.read_json(path_to_test_set, orient="records")
idx = [ 3,  12,  15,  34,  66,  67,  68,  69,  71,  72,  73, 102, 109,
       113, 115, 116, 119, 144, 150, 159, 160, 193, 195, 198]

idx1 = [67,  68,  69,  71,  72,  73, 102, 109]
question = dd.iloc[idx1]

for index, item in question.iterrows():
    query = item['query']
    actual_answer = item['answer']
    print(f"Query: {query}, actual answer: {actual_answer}")
    try:

        answer = graph_retriever.answer(query, verbose=True)
        print(f'Graph RAG answer: {answer}')
    except Exception as e:
        print(f"Graph RAG Error: {str(e)}")
    

In [ ]:
import pandas as pd
from src.bots.graph_retrieval_v3 import GraphRAGRetriever
from src.configs.config import settings
from src.db.graph_store import graph_db
from src.db.vector_store import chroma_db
from src.bots.hybrid_retrieval import HybridGraphVectorRetriever

graph_retriever = GraphRAGRetriever(graph_db, settings.OPENAI_API_KEY)
# hybrid_retriever = HybridGraphVectorRetriever(graph_db, chroma_db, settings.OPENAI_API_KEY)
path_to_test_set = "./results/test_set.json"
dd = pd.read_json(path_to_test_set, orient="records")
idx = [ 3,  12,  15,  34,  66,  67,  68,  69,  71,  72,  73, 102, 109,
       113, 115, 116, 119, 144, 150, 159, 160, 193, 195, 198]
# dd.loc[idx, 'graph_rag_status'] = 'PENDING'
dd

In [ ]:
dd.to_json(path_to_test_set, orient="records", indent=2)